In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from typing import Optional
import plotly.express as px
import plotly.graph_objects as go

In [2]:
# Get data into dataframe
df = pd.read_csv('../WMA_fractions_v2.csv', skiprows=1)

# Preprocess data to only have temperature, salinity and dissolved oxygen
df_TSO = df.copy()
df_TSO = df_TSO[['Conservative_Temperature_[deg_C]', 'Absolute_Salinity_[PSU]', 'Dissolved_Oxygen_[micro_mol_per_kg]','Depth_[m]', 'Latitude_[deg_N]', 'Longitude_[deg_E]']]
df_TSO = df_TSO.dropna()

In [3]:
def uniform_geographic_sample(
    df: pd.DataFrame,
    n_samples: int,
    lat_col: str = 'latitude',
    lon_col: str = 'longitude',
    n_grid_cells: int = 50,
    random_state: Optional[int] = None) -> pd.DataFrame:
    """
    Sample points from a dataset to minimize spatial density variability.
    
    Uses grid-based stratified sampling to ensure uniform geographic coverage.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Input dataframe with geographic coordinates
    n_samples : int
        Number of samples to draw
    lat_col : str
        Name of the latitude column
    lon_col : str
        Name of the longitude column
    n_grid_cells : int
        Number of grid cells per dimension (creates n_grid_cells x n_grid_cells grid)
    random_state : int, optional
        Random seed for reproducibility
        
    Returns:
    --------
    pd.DataFrame
        Sampled dataframe with uniform spatial distribution
    """
    if random_state is not None:
        np.random.seed(random_state)
    
    df = df.copy()
    
    # Get coordinate bounds
    lat_min, lat_max = df[lat_col].min(), df[lat_col].max()
    lon_min, lon_max = df[lon_col].min(), df[lon_col].max()
    
    # Create grid cells
    lat_bins = np.linspace(lat_min, lat_max, n_grid_cells + 1)
    lon_bins = np.linspace(lon_min, lon_max, n_grid_cells + 1)
    
    # Assign each point to a grid cell
    df['_lat_bin'] = pd.cut(df[lat_col], bins=lat_bins, labels=False, include_lowest=True)
    df['_lon_bin'] = pd.cut(df[lon_col], bins=lon_bins, labels=False, include_lowest=True)
    df['_grid_cell'] = df['_lat_bin'].astype(str) + '_' + df['_lon_bin'].astype(str)
    
    # Count points per grid cell
    cell_counts = df['_grid_cell'].value_counts()
    occupied_cells = len(cell_counts)
    
    # Calculate target samples per cell for uniform distribution
    samples_per_cell = n_samples / occupied_cells
    
    # Strategy: Sample equally from each occupied cell for uniform coverage
    # First pass: try to take equal samples from each cell
    base_samples_per_cell = n_samples // occupied_cells
    extra_samples = n_samples % occupied_cells
    
    sampled_dfs = []
    cells_to_boost = []
    
    for i, cell_id in enumerate(cell_counts.index):
        cell_df = df[df['_grid_cell'] == cell_id]
        
        # Base samples for this cell
        n_from_cell = base_samples_per_cell
        
        # Distribute extra samples to first few cells
        if i < extra_samples:
            n_from_cell += 1
        
        # Can't sample more than available in cell
        n_from_cell = min(n_from_cell, len(cell_df))
        
        if n_from_cell > 0:
            sampled = cell_df.sample(n=n_from_cell, replace=False)
            sampled_dfs.append(sampled)
        
        # Track if this cell couldn't provide enough samples
        if n_from_cell < base_samples_per_cell + (1 if i < extra_samples else 0):
            deficit = (base_samples_per_cell + (1 if i < extra_samples else 0)) - n_from_cell
            cells_to_boost.append(deficit)
    
    # Combine all samples
    result = pd.concat(sampled_dfs, ignore_index=True)
    
    # If we have a deficit, sample more from cells that have remaining points
    if len(result) < n_samples:
        remaining_df = df[~df.index.isin(result.index)]
        if len(remaining_df) > 0:
            additional_needed = n_samples - len(result)
            additional = remaining_df.sample(n=min(additional_needed, len(remaining_df)), replace=False)
            result = pd.concat([result, additional], ignore_index=True)
    
    # Remove helper columns
    result = result.drop(columns=['_lat_bin', '_lon_bin', '_grid_cell'])
    
    return result

In [4]:
df_sampled = uniform_geographic_sample(df_TSO, n_samples=100000, lat_col='Latitude_[deg_N]', lon_col='Longitude_[deg_E]', n_grid_cells=100, random_state=22)

In [5]:
# Extract values into a tensor
X = torch.tensor(df_sampled.values, dtype=torch.float32)
dataset = TensorDataset(X)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

In [6]:
# Define Variational Autoencoder

class VAE(nn.Module):
    def __init__(self, input_dim=3, latent_dim=2):
        super().__init__()

        # Encoder
        self.fc1 = nn.Linear(input_dim, 16)
        self.fc_mu = nn.Linear(16, latent_dim)
        self.fc_logvar = nn.Linear(16, latent_dim)

        # Decoder
        self.fc2 = nn.Linear(latent_dim, 16)
        self.fc3 = nn.Linear(16, input_dim)

    def encode(self, x):
        h = F.relu(self.fc1(x))
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = F.relu(self.fc2(z))
        return self.fc3(h)     # no sigmoid for real-valued data

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

In [7]:
# Define loss function (mse and KL divergence)
def vae_loss(recon_x, x, mu, logvar):
    mse = F.mse_loss(recon_x, x, reduction='sum')
    kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return mse + kld

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"

n_models = 5
reps = []

while len(reps) < n_models:

    model_idx = len(reps)
    print(f"\nTraining model {model_idx+1}/{n_models}")

    # Loop until one successful training run completes
    successful = False

    while not successful:

        # Create a fresh model + optimizer
        model = VAE(input_dim=6, latent_dim=2).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

        epochs = 50
        nan_found = False

        for epoch in range(epochs):
            model.train()
            total_loss = 0.0

            for batch in loader:
                x = batch[0].to(device)

                optimizer.zero_grad()

                recon, mu, logvar = model(x)
                loss = vae_loss(recon, x, mu, logvar)

                # Detect NaN inside the loss tensor
                if torch.isnan(loss):
                    nan_found = True
                    print(f"[Epoch {epoch}] Loss is NaN → restarting model.")
                    break

                loss.backward()
                optimizer.step()

                total_loss += loss.item()

            if nan_found:
                break  # break out of epoch loop → restart new model

            # Detect NaN in aggregated loss
            if math.isnan(total_loss):
                nan_found = True
                print(f"[Epoch {epoch}] Total loss is NaN → restarting model.")
                break

        # Check if this model finished without NaN
        if not nan_found:
            successful = True
            print("Model trained successfully (no NaNs).")

    # Extract representation after a successful training
    model.eval()
    with torch.no_grad():
        mu, logvar = model.encode(X.to(device))
        z = mu.cpu()

    reps.append(z)


Training model 1/5
[Epoch 0] Loss is NaN → restarting model.
[Epoch 0] Loss is NaN → restarting model.
Model trained successfully (no NaNs).

Training model 2/5
[Epoch 0] Loss is NaN → restarting model.
Model trained successfully (no NaNs).

Training model 3/5
Model trained successfully (no NaNs).

Training model 4/5
Model trained successfully (no NaNs).

Training model 5/5
Model trained successfully (no NaNs).


# Gap statistic

In [11]:
import numpy as np
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

def gap_statistic_gmm(X, k_max=10, B=20, random_state=22):

    # Torch → numpy
    if hasattr(X, "detach"):
        X = X.detach().cpu().numpy()

    if X.ndim != 2:
        raise ValueError(f"X must be 2D, got shape {X.shape}")

    # Force float64
    X = X.astype(np.float64)

    # Scale data
    X = StandardScaler().fit_transform(X)

    np.random.seed(random_state)

    n, d = X.shape
    mins = X.min(axis=0)
    maxs = X.max(axis=0)

    gaps = np.full(k_max, np.nan)
    sk = np.full(k_max, np.nan)

    for k in range(1, k_max + 1):

        # Real data
        try:
            gmm = GaussianMixture(
                n_components=k,
                covariance_type="full",
                reg_covar=1e-5,
                n_init=5,
                random_state=random_state
            )
            gmm.fit(X)
            Wk = -gmm.score(X) * n
        except ValueError:
            continue

        # Reference data
        Wk_refs = []

        for b in range(B):
            X_ref = np.random.uniform(mins, maxs, size=(n, d))

            try:
                gmm_ref = GaussianMixture(
                    n_components=k,
                    covariance_type="full",
                    reg_covar=1e-5,
                    n_init=5,
                    random_state=random_state
                )
                gmm_ref.fit(X_ref)
                Wk_refs.append(-gmm_ref.score(X_ref) * n)
            except ValueError:
                continue

        # Need enough successful reference fits
        if len(Wk_refs) < max(3, B // 2):
            continue

        log_Wk_refs = np.log(Wk_refs)

        gaps[k - 1] = np.mean(log_Wk_refs) - np.log(Wk)
        sk[k - 1] = np.sqrt(1 + 1 / len(Wk_refs)) * np.std(log_Wk_refs)

    return gaps, sk

In [12]:
for i, rep in enumerate(reps):
    gaps, sk = gap_statistic_gmm(rep, k_max=10)

    for k in range(len(gaps) - 1):
        if gaps[k] >= gaps[k + 1] - sk[k + 1]:
            optimal_k = k + 1
            break

    print("Optimal k for rep", i, ": ", optimal_k)

Optimal k for rep 0 :  8


/var/folders/v0/0lp_zzrd33sgh7y2jyh9zzdm0000gn/T/ipykernel_29009/2637708239.py:70: RuntimeWarning: invalid value encountered in log
  gaps[k - 1] = np.mean(log_Wk_refs) - np.log(Wk)
/var/folders/v0/0lp_zzrd33sgh7y2jyh9zzdm0000gn/T/ipykernel_29009/2637708239.py:70: RuntimeWarning: invalid value encountered in log
  gaps[k - 1] = np.mean(log_Wk_refs) - np.log(Wk)
/var/folders/v0/0lp_zzrd33sgh7y2jyh9zzdm0000gn/T/ipykernel_29009/2637708239.py:70: RuntimeWarning: invalid value encountered in log
  gaps[k - 1] = np.mean(log_Wk_refs) - np.log(Wk)
/var/folders/v0/0lp_zzrd33sgh7y2jyh9zzdm0000gn/T/ipykernel_29009/2637708239.py:70: RuntimeWarning: invalid value encountered in log
  gaps[k - 1] = np.mean(log_Wk_refs) - np.log(Wk)
/var/folders/v0/0lp_zzrd33sgh7y2jyh9zzdm0000gn/T/ipykernel_29009/2637708239.py:70: RuntimeWarning: invalid value encountered in log
  gaps[k - 1] = np.mean(log_Wk_refs) - np.log(Wk)
/var/folders/v0/0lp_zzrd33sgh7y2jyh9zzdm0000gn/T/ipykernel_29009/2637708239.py:70: Runtime

Optimal k for rep 1 :  8
Optimal k for rep 2 :  6
Optimal k for rep 3 :  9
Optimal k for rep 4 :  6


# Slope Heuristics/Jump Method